# GA Optimization: Coupled Fuzzy Beats Both Baselines (GA-01..05)

50x50 real-valued GA (`src/genetic_algorithm.py`, 62-D: 60 MF genes +
`k_heat`/`k_vent` coupling gains) evolves on seed-7 weather; the artifact
`results/ga_optimized_params.json` (schema v2) is the only data source here.
Holdout proof runs all four controllers on FRESH seed-21 weather the GA never
saw. Per D-06/Amendment A2 the comparison is **intelligent-coupled
(GA fuzzy) vs classical-decoupled (on/off, PID, hand fuzzy — sealed)**:
humidity error drives a heater/vent boost after the untouched decoupled loops.



In [ ]:
import json
import os
import tempfile
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from src.greenhouse_model import (step, load_weather_profile, daytime_solar,
                                  params, relative_humidity, wsat_g_per_kg)
from src.weather_profile_generator import generate_weather_profile
from src.onoff_controller import onoff_step
from src.pid_controller import pid_step
from src.fuzzy_controller import fuzzy_step, FUZZY_MFS, tri
from src.ga_fuzzy_controller import ga_fuzzy_step, load_optimized_mfs
from src.genetic_algorithm import W1, W2, W3  # locked weights: proof must match fitness



In [ ]:
# Train weather is the immutable evolution input: load it, never regenerate
# (generate at seed 7 only if the CSV is missing). The JSON artifact is the
# ONLY data source for tuned params + history — no evolution runs inline.
import os
CSV = "weather/weather_profile.csv"
if not os.path.exists(CSV):
    generate_weather_profile(CSV, seed=7)

t_min, T_out, w_out = load_weather_profile(CSV)
hours = np.arange(0, 1440) / 60.0          # dt = 60 s -> one step per minute
T_out_hr = np.interp(hours, t_min, T_out)
w_out_hr = np.interp(hours, t_min, w_out)

JSON = "results/ga_optimized_params.json"
with open(JSON) as _f:
    d = json.load(_f)
assert d["schema_version"] == 2 and d["population"] == 50 and d["generations"] == 50, \
    "stale artifact: expected run-3 schema v2 50x50"
mfs_opt, gains_opt = load_optimized_mfs(JSON)
print("schema v%s" % d.get("schema_version", 1),
      "| train fit %.2f" % d["fitness"], "| coupling", d.get("coupling"))
print("history gens", len(d["history"]["best"]))



In [ ]:
# Holdout proof: FRESH seed-21 weather on an isolated temp path (never
# passed to fitness), all four controllers on that same shared weather.
# Closed-loop runner: T/w init 20.0 / 0.65*wsat(20), setpoints as args.
T_SP, RH_SP, DT = 25.0, 70.0, 60.0

holdout_csv = os.path.join(tempfile.mkdtemp(), "holdout_21.csv")
generate_weather_profile(holdout_csv, seed=21)
h_min, h_T, h_w = load_weather_profile(holdout_csv)
H_T = np.interp(hours, h_min, h_T)
H_w = np.interp(hours, h_min, h_w)

def run_closed_loop(ctl):
    T, w, st = 20.0, 0.65 * wsat_g_per_kg(20.0), None
    Ts, RHs, iae_T, iae_H, E = [], [], 0.0, 0.0, 0.0
    for i, h in enumerate(hours):
        rh = relative_humidity(w, T)
        act, st = ctl(T, rh, st)
        T, w, rh = step(DT, act, (H_T[i], H_w[i],
                                  daytime_solar(h, params["A_sol"])), T, w)
        Ts.append(T)
        RHs.append(rh)
        iae_T += abs(T - T_SP) / 60.0
        iae_H += abs(rh - RH_SP) / 60.0
        E += sum(act.values()) / 60.0
    return np.array(Ts), np.array(RHs), iae_T, iae_H, E

runs = {
    "on/off": run_closed_loop(lambda T, rh, st: onoff_step(T, rh, T_SP, RH_SP, st)),
    "PID": run_closed_loop(lambda T, rh, st: pid_step(T, rh, T_SP, RH_SP, st, DT)),
    "hand fuzzy": run_closed_loop(lambda T, rh, st: fuzzy_step(T, rh, T_SP, RH_SP, st, DT, FUZZY_MFS)),
    "GA fuzzy": run_closed_loop(lambda T, rh, st: ga_fuzzy_step(T, rh, T_SP, RH_SP, st, DT, JSON)),
}
costs = {}
for name, (Ts, RHs, iae_T, iae_H, E) in runs.items():
    costs[name] = W1 * iae_T + W2 * iae_H + W3 * E
    print("%-10s IAE_T %7.2f IAE_H %7.2f E %6.2f cost %8.2f"
          % (name, iae_T, iae_H, E, costs[name]))
# Verdict: RELATIVE ordering only — GA below on/off AND below PID, no absolutes.
assert costs["GA fuzzy"] < costs["on/off"], (costs["GA fuzzy"], costs["on/off"])
assert costs["GA fuzzy"] < costs["PID"], (costs["GA fuzzy"], costs["PID"])
print("verdict: GA < on/off AND GA < PID on shared seed-21 holdout — GA-05 HOLDS")



**Verdict framing (D-06/A2):** the wet-side gap that blocked runs 1-2
(GA ~635 vs on/off ~514, 100% wet-side per the split probe) is closed by
structure, not tuning — cross-coupling gives the GA fuzzy dehumidification
authority (heater + vent when humid) that independent loops cannot express.
The verdict cell above asserts the relative bar only: GA cost below on/off
AND below PID on the same shared seed-21 weather, with per-term
IAE_T/IAE_H/E printed — no absolute threshold anywhere.



In [ ]:
# Convergence (best/mean per generation from the artifact) + holdout
# 4-controller T/RH traces + MF before/after overlays, saved beside the nb.
fig, ax = plt.subplots(2, 2, figsize=(12, 8))
gens = np.arange(1, len(d["history"]["best"]) + 1)
ax[0, 0].plot(gens, d["history"]["best"], label="best")
ax[0, 0].plot(gens, d["history"]["mean"], label="mean", alpha=0.7)
ax[0, 0].set_xlabel("generation")
ax[0, 0].set_ylabel("train weighted cost")
ax[0, 0].set_title("GA convergence (50x50, seed 0, train seed-7)")
ax[0, 0].legend()
for name, (Ts, RHs, _a, _b, _c) in runs.items():
    ax[0, 1].plot(hours, Ts, label=name, linewidth=1.2)
ax[0, 1].axhline(T_SP, color="k", linestyle="--", linewidth=0.8)
ax[0, 1].set_xlabel("hour")
ax[0, 1].set_ylabel("T (degC)")
ax[0, 1].set_title("Holdout T traces (seed-21, shared)")
ax[0, 1].legend(fontsize=8)
for name, (Ts, RHs, _a, _b, _c) in runs.items():
    ax[1, 0].plot(hours, RHs, label=name, linewidth=1.2)
ax[1, 0].axhline(RH_SP, color="k", linestyle="--", linewidth=0.8)
ax[1, 0].set_xlabel("hour")
ax[1, 0].set_ylabel("RH (%)")
ax[1, 0].set_title("Holdout RH traces (seed-21, shared)")
ax[1, 0].legend(fontsize=8)
grid = np.linspace(-40, 40, 400)  # eH universe: hand vs GA-tuned shapes
for c, w in zip(FUZZY_MFS["eH"]["centers"], FUZZY_MFS["eH"]["widths"]):
    ax[1, 1].plot(grid, [tri(x, c, w) for x in grid], "C0-", alpha=0.5)
for c, w in zip(mfs_opt["eH"]["centers"], mfs_opt["eH"]["widths"]):
    ax[1, 1].plot(grid, [tri(x, c, w) for x in grid], "C1--", alpha=0.8)
ax[1, 1].plot([], [], "C0-", label="hand eH")
ax[1, 1].plot([], [], "C1--", label="GA eH")
ax[1, 1].set_title("MF before/after: eH (k_heat %.3f, k_vent %.3f)"
                   % (gains_opt["k_heat"], gains_opt["k_vent"]))
ax[1, 1].legend(fontsize=8)
fig.tight_layout()
fig.savefig("notebooks/04_ga_optimization.png", dpi=100)
print("saved notebooks/04_ga_optimization.png")

